# Tutorial: Polars & Dataframely

In [8]:
%reload_ext autoreload
%autoreload 2

In [10]:
import polars as pl

## Task 1: Exploring the Raw Data

In [11]:
policies = pl.read_parquet("data/policies.parquet")
models = pl.read_parquet("data/models.parquet")

In [14]:
# TODO: Print data frames to explore their structure
print(policies)
print(models)

shape: (1_000_000, 7)
┌───────┬───────────────┬───────────────┬────────────┬───────────────┬──────────────┬──────────────┐
│ model ┆ policy_id     ┆ policy_tenure ┆ age_of_car ┆ age_of_policy ┆ area_cluster ┆ population_d │
│ ---   ┆ ---           ┆ ---           ┆ ---        ┆ holder        ┆ ---          ┆ ensity       │
│ str   ┆ str           ┆ f64           ┆ f64        ┆ ---           ┆ str          ┆ ---          │
│       ┆               ┆               ┆            ┆ f64           ┆              ┆ i64          │
╞═══════╪═══════════════╪═══════════════╪════════════╪═══════════════╪══════════════╪══════════════╡
│ M1080 ┆ policy6153536 ┆ 1.058001      ┆ 29.806413  ┆ 42.86828      ┆ C9           ┆ 17865        │
│ M791  ┆ policy6592089 ┆ 1.05291       ┆ 12.161883  ┆ 29.440503     ┆ C21          ┆ 3268         │
│ M880  ┆ policy2173464 ┆ 0.767566      ┆ 31.490837  ┆ 49.018794     ┆ C5           ┆ 34771        │
│ M868  ┆ policy8580874 ┆ 1.058158      ┆ 4.127161   ┆ 26.550939     

In [15]:
print(f"policies: {policies.estimated_size('mb'):.2f} MB")
print(f"models:   {models.estimated_size('mb'):.2f} MB")

policies: 48.96 MB
models:   0.26 MB


In [ ]:
# TODO: Explore the contents of the columns:
#  - Find the unique values of `fuel_type`. How many are there?
print(models.select(pl.col("fuel_type").unique().sort()))
print(f"n unique fuel types: {models['fuel_type'].n_unique()}")
#  - Find the range of values of `airbags`. What would be an appropriate data type for this column?
#  - Check out the values found in the `is_` columns of the models. Can you verify that all rows
#    in these columns contain exactly these values?
#  - Check out the `max_power` and `max_torque` columns of the models. Do they follow a consistent
#    format?



# Hints:
# * You can use `df.select("column")` to see a single column, or write expressions on it like `df.select(pl.col("column").unique())`
# * You can use `df.group_by("column").len()` to do a simple group-by-and-count
# * group-bys can also evaluate more complicated expressions like `df.group_by("column").agg(name=my_expression)`, where `my_expression` could be `pl.col("column").max()`, or `pl.col("some_other_colum").mean()`

shape: (3, 1)
┌───────────┐
│ fuel_type │
│ ---       │
│ str       │
╞═══════════╡
│ CNG       │
│ Diesel    │
│ Petrol    │
└───────────┘
n unique fuel types: 3


## Task 2: Preprocessing the Raw Data

In [20]:
from pipeline.data import RawData
from pipeline.preprocess import preprocess

In [21]:
raw_data = RawData(models, policies)
preprocessed = preprocess(raw_data)

In [22]:
# Memory footprint after preprocessing
for label, raw_df, prep_df in (
    ("policies", policies, preprocessed.policies),
    ("models", models, preprocessed.models),
):
    print(
        f"{label}: raw {raw_df.estimated_size('mb'):.2f} MB → "
        f"preprocessed {prep_df.estimated_size('mb'):.2f} MB"
    )

# Physical storage for fuel_type: raw strings vs enum codes after preprocessing
print("fuel_type (raw) physical:", models["fuel_type"].to_physical())
print("fuel_type (prep) physical:", preprocessed.models["fuel_type"].to_physical())

policies: raw 48.96 MB → preprocessed 30.52 MB
models: raw 0.26 MB → preprocessed 0.08 MB
fuel_type (raw) physical: shape: (1_228,)
Series: 'fuel_type' [str]
[
	"Diesel"
	"CNG"
	"Petrol"
	"Diesel"
	"Petrol"
	…
	"Petrol"
	"Petrol"
	"Petrol"
	"CNG"
	"Petrol"
]
fuel_type (prep) physical: shape: (1_228,)
Series: 'fuel_type' [u8]
[
	1
	0
	2
	1
	2
	…
	2
	2
	2
	0
	2
]


## Task 3: Generate a Report from the Raw Data

Complete **Task 3** in [`pipeline/report.py`](pipeline/report.py): `find_three_most_popular_make_and_models`, `find_safest_models`, and `find_average_car_volume_by_age`. This repo includes a reference implementation you can open beside the notebook.

In [23]:
from pipeline.report import build_report

report = build_report(
    prep=preprocessed
)

In [24]:
print(report.to_string())


                           REPORT                           

Top 3 Most Popular Make/Model Combinations:
------------------------------------------------------------
shape: (3, 3)
┌──────┬───────┬────────┐
│ make ┆ model ┆ count  │
│ ---  ┆ ---   ┆ ---    │
│ i64  ┆ cat   ┆ u32    │
╞══════╪═══════╪════════╡
│ 10   ┆ M868  ┆ 263656 │
│ 6    ┆ M652  ┆ 131735 │
│ 9    ┆ M573  ┆ 76174  │
└──────┴───────┴────────┘
Top 5 Safest Models:
------------------------------------------------------------
shape: (5, 3)
┌───────┬─────────┬──────────────┐
│ model ┆ segment ┆ safety_score │
│ ---   ┆ ---     ┆ ---          │
│ cat   ┆ cat     ┆ u16          │
╞═══════╪═════════╪══════════════╡
│ M282  ┆ A       ┆ 21           │
│ M626  ┆ C1      ┆ 21           │
│ M919  ┆ A       ┆ 21           │
│ M998  ┆ Utility ┆ 21           │
│ M194  ┆ C2      ┆ 20           │
└───────┴─────────┴──────────────┘


Average Car Volume by Age:
------------------------------------------------------------
shape: (5, 3)

# Task 4 (optional): Run the entire pipeline in lazy mode

In [ ]:
import time

import polars as pl
from IPython.display import Image, display

from pipeline.data import RawData
from pipeline.orchestration import run_pipeline

models_path = "data/models.parquet"
policies_path = "data/policies.parquet"

# Eager: materialize Parquet up front, then run the full pipeline and render the report
t0 = time.perf_counter()
raw_eager = RawData(
    models=pl.read_parquet(models_path),
    policies=pl.read_parquet(policies_path),
)
report_eager = run_pipeline(raw_eager)
_ = report_eager.to_string()
t_eager = time.perf_counter() - t0
print(f"Eager (read_parquet + run_pipeline + to_string): {t_eager:.4f} s")

# Lazy: scans stay deferred until `to_string` collects the LazyFrames inside `Report`
t0 = time.perf_counter()
raw_lazy = RawData(
    models=pl.scan_parquet(models_path),
    policies=pl.scan_parquet(policies_path),
)
report_lazy = run_pipeline(raw_lazy)
_ = report_lazy.to_string()
t_lazy = time.perf_counter() - t0
print(f"Lazy (scan_parquet + run_pipeline + to_string): {t_lazy:.4f} s")

# Query plan for `report_lazy.volume` (needs Graphviz `dot` + matplotlib in the kernel env)
lv = report_lazy.volume
assert isinstance(lv, pl.LazyFrame)
print("\n--- volume: explain (optimized) ---")
print(lv.explain())

png_opt = "task4_volume_plan_optimized.png"
png_raw = "task4_volume_plan_unoptimized.png"

print("\n--- volume: query plan images (saved next to this notebook) ---")
lv.show_graph(optimized=True, show=False, output_path=png_opt)
display(Image(filename=png_opt))

lv.show_graph(optimized=False, show=False, output_path=png_raw)
display(Image(filename=png_raw))


# Task 5: using dataframely schemas for raw data

In [ ]:
# TODO:
# - In `pipeline.schema.raw`, there are basic dataframely schemas for the raw data.
#   Import them and validate the data. 
# - Validation for `models` will fail. Use Schema.filter to to separate the good rows from the bad rows. Use `FailureInfo.details()` to inspect the failing rows.
# - Change the schema such that the raw data validation stops failing.
# - Remember the problem so we can fix it in preprocessing in the next step.

# Task 6: using dataframely schemas for preprocessed data

In [ ]:
# TODO:
# - In `pipeline.schema.preprocessed`, there are similar schema definitions for our preprocessed data.
#   Extend the schemas such that they catch the mm-vs-cm issue we encountered when calculcating the car volume above.
# - Verify that validating the preprocessed data fails, and extend the preprocessing code such that validation succeeds again.
# - Fix the preprocessing code for `models` to deal with the validation problem from the last step.
# - Add `Schema.validate(cast=True, eager=False)` calls to the preprocessing methods and adapt the type hints

# Task 7: Using dataframely schemas for report outputs

In [ ]:
# TODO:
# - Add dataframely schemas for the tables we create in `report.py`. The schemas go in `pipeline/schema/report.py`
# - Extend the functions with the right type hints and add calls to `.validate`

# Task 8: Using dataframely to write a test for our report

In [ ]:
# TODO:
# - Fill in the code in `tests/test_report.py`. Execute the test via `pixi run pytest tests/test_report.py`